In [ ]:
import json
import time
import textwrap
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional

from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_warehouse('SNOWFLAKE_LEARNING_WH')

# ── Configuration ────────────────────────────────────────────────────────────
MODEL       = 'claude-sonnet-4-5'         # hosted by Snowflake — no local GPU
KB_DB       = 'CLINICAL_ROUNDS_DEMO'      # knowledge base database
KB_SCHEMA   = 'PATIENT_DATA'             # knowledge base schema
MAX_TOKENS  = 2048

print(f'Session: {session.get_current_account()}')
print(f'LLM model: {MODEL}')
print(f'Knowledge base: {KB_DB}.{KB_SCHEMA}')

In [ ]:
import base64
import requests
from pathlib import Path

class CortexRESTClient:
    """Cortex inference client via session token. Supports text + vision."""

    def __init__(self, host: str, token: str, model: str):
        self.host = host
        self.model = model
        self.url = f'https://{host}/api/v2/cortex/v1/chat/completions'
        self.headers = {
            'Authorization': f'Snowflake Token="{token}"',
            'Content-Type': 'application/json',
            'Accept': 'application/json',
        }

    def __repr__(self):
        return f'CortexRESTClient(model={self.model!r})'

    @classmethod
    def from_session(cls, session, model: str):
        conn = session.connection
        host = conn.host.replace('_', '-')
        token = conn.rest._token
        return cls(host, token, model)

    def complete(self, prompt: str, system_prompt: str = '',
                 max_tokens: int = 2048, temperature: float = 0.3) -> str:
        messages = []
        if system_prompt:
            messages.append({'role': 'system', 'content': system_prompt})
        messages.append({'role': 'user', 'content': prompt})
        return self._call(messages, max_tokens, temperature)

    def complete_with_image(self, prompt: str, image_source: str,
                            system_prompt: str = '',
                            max_tokens: int = 2048, temperature: float = 0.3) -> str:
        """Multimodal: pass a file path, presigned URL, or base64 string."""
        image_url = self._resolve_image(image_source)
        content = [
            {'type': 'text', 'text': prompt},
            {'type': 'image_url', 'image_url': {'url': image_url}},
        ]
        messages = []
        if system_prompt:
            messages.append({'role': 'system', 'content': system_prompt})
        messages.append({'role': 'user', 'content': content})
        return self._call(messages, max_tokens, temperature)

    def _call(self, messages: list, max_tokens: int, temperature: float) -> str:
        payload = {
            'model': self.model,
            'messages': messages,
            'max_completion_tokens': max_tokens,
            'temperature': temperature,
        }
        response = requests.post(self.url, headers=self.headers, json=payload, timeout=120)
        if response.status_code != 200:
            raise RuntimeError(f'Cortex API error {response.status_code}: {response.text[:300]}')

        content = ''
        for line in response.text.split('\n'):
            if line.startswith('data: '):
                chunk = line[6:]
                if chunk.strip() == '[DONE]':
                    break
                try:
                    data = json.loads(chunk)
                    delta = data.get('choices', [{}])[0].get('delta', {})
                    content += delta.get('content', '')
                except json.JSONDecodeError:
                    continue
        if not content:
            try:
                data = json.loads(response.text)
                content = data['choices'][0]['message']['content']
            except (json.JSONDecodeError, KeyError, IndexError):
                pass
        return content

    @staticmethod
    def _resolve_image(image_source: str) -> str:
        if image_source.startswith(('https://', 'http://', 'data:')):
            return image_source
        path = Path(image_source)
        if path.exists():
            data = base64.b64encode(path.read_bytes()).decode()
            suffix = path.suffix.lower().lstrip('.')
            mime = {'png': 'image/png', 'jpg': 'image/jpeg',
                    'jpeg': 'image/jpeg', 'gif': 'image/gif',
                    'webp': 'image/webp'}.get(suffix, 'image/png')
            return f'data:{mime};base64,{data}'
        return f'data:image/png;base64,{image_source}'


rest_client = CortexRESTClient.from_session(session, MODEL)
print(rest_client)
print(rest_client.complete('Reply with exactly: OK'))

In [ ]:
def make_rest_backend(rest_client: CortexRESTClient):
    """Monkey-patch a subagent's _complete() to use the REST API instead of SQL."""
    def _rest_complete(self_agent, user_prompt: str, extra_system: str = '') -> str:
        system = self_agent.system_prompt
        if extra_system:
            system = f'{system}\n\n{extra_system}'
        return rest_client.complete(user_prompt, system_prompt=system)
    return _rest_complete


def use_rest_backend(agent, rest_client: CortexRESTClient):
    """Switch a single subagent to the REST API backend."""
    import types
    agent._complete = types.MethodType(make_rest_backend(rest_client), agent)
    print(f'{agent.__class__.__name__} → REST API backend')


def use_sql_backend(agent):
    """Restore a subagent to the original SQL (CORTEX.COMPLETE) backend."""
    try:
        del agent._complete
    except AttributeError:
        pass
    print(f'{agent.__class__.__name__} → SQL backend (CORTEX.COMPLETE)')


# ── Switch all agents to REST ───────────────────────────────────────────────
_agent_names = ['planner', 'searcher', 'writer', 'reflector', 'qa_agent']
_agents = [globals()[n] for n in _agent_names if n in globals()]

if rest_client and _agents:
    for ag in _agents:
        use_rest_backend(ag, rest_client)
elif not rest_client:
    print('REST client not available — all agents using SQL backend.')
else:
    print('Agents not yet defined — run this cell again after defining them.')

In [ ]:
class CoCoSubagentRouter:
    """Routes research tasks to specialised subagents with security envelopes.

    Mirrors the routing + envelope design from subagent-cortex-code:
      - Semantic routing via LLM (not keyword matching)
      - Envelopes: RO | RW | RESEARCH | DEPLOY
      - Approval modes: prompt | auto | envelope_only
    """

    # Security envelopes — what each mode allows
    ENVELOPES = {
        'RO':       'Read-only: search and retrieve information',
        'RW':       'Read-Write: create and modify content',
        'RESEARCH': 'Research mode: explore, analyse, synthesise',
        'DEPLOY':   'Deploy mode: finalise and publish output',
    }

    # Allowed subagent types per envelope
    ENVELOPE_AGENTS = {
        'RO':       {'searcher', 'qa'},
        'RW':       {'writer', 'reflector'},
        'RESEARCH': {'planner', 'searcher', 'reflector', 'qa'},
        'DEPLOY':   {'writer', 'planner'},
    }

    def __init__(self, session, model: str, approval_mode: str = 'prompt'):
        self.session = session
        self.model = model
        self.approval_mode = approval_mode  # prompt | auto | envelope_only
        self._audit_log: list[dict] = []

    # ── Cortex call ──────────────────────────────────────────────────────────
    def _complete(self, prompt: str, system: str = '') -> str:
        full = f"{system}\n\n{prompt}" if system else prompt
        # Escape single quotes for SQL
        escaped = full.replace("'", "''")
        row = self.session.sql(
            f"SELECT SNOWFLAKE.CORTEX.COMPLETE('{self.model}', '{escaped}')"
        ).collect()
        return row[0][0].strip() if row else ''

    # ── Semantic routing ─────────────────────────────────────────────────────
    def route(self, request: str) -> dict:
        """Classify request intent into one of five subagent types."""
        system = textwrap.dedent("""
            You are a routing classifier for a biomedical research agent.
            Given a user request, output ONLY a JSON object with:
              - "agent": one of [planner, searcher, writer, reflector, qa]
              - "envelope": one of [RO, RW, RESEARCH, DEPLOY]
              - "confidence": 0-100
              - "reason": one sentence

            Rules:
            - planner: creating a research plan or outline
            - searcher: retrieving facts from clinical data
            - writer: drafting or writing report sections
            - reflector: identifying gaps or generating follow-up queries
            - qa: answering questions about an existing report

            Envelope rules:
            - RO for searcher/qa
            - RW for writer
            - RESEARCH for planner/reflector
            - DEPLOY only when explicitly finalising

            Output ONLY the JSON object, no markdown fences.
        """)
        raw = self._complete(request, system)
        try:
            # Strip any accidental markdown fences
            clean = raw.strip().lstrip('```json').lstrip('```').rstrip('```')
            return json.loads(clean)
        except json.JSONDecodeError:
            return {'agent': 'searcher', 'envelope': 'RO',
                    'confidence': 50, 'reason': 'fallback routing'}

    # ── Envelope enforcement ─────────────────────────────────────────────────
    def execute_with_envelope(
        self, agent_fn, prompt: str, envelope: str = 'RESEARCH'
    ) -> str:
        """Execute an agent function inside a security envelope."""
        if envelope not in self.ENVELOPES:
            raise ValueError(f'Unknown envelope: {envelope}. Use: {list(self.ENVELOPES)}')

        entry = {
            'ts': time.strftime('%Y-%m-%dT%H:%M:%S'),
            'envelope': envelope,
            'prompt_preview': prompt[:120],
        }

        if self.approval_mode == 'prompt':
            print(f'\n🔐 Cortex Subagent execution request:')
            print(f'   Envelope : {envelope} — {self.ENVELOPES[envelope]}')
            print(f'   Preview  : {prompt[:100]}...')
            print(f'   Mode     : {self.approval_mode}')
            decision = input('   Approve? [yes/no]: ').strip().lower()
            if decision not in ('yes', 'y'):
                return '[Execution denied by user]'
        elif self.approval_mode in ('auto', 'envelope_only'):
            # Auto-approved — log and proceed
            pass

        result = agent_fn(prompt)
        entry['status'] = 'ok'
        self._audit_log.append(entry)
        return result

    def audit_summary(self):
        print(f'\n📋 Audit log ({len(self._audit_log)} executions):')
        for e in self._audit_log:
            print(f"  [{e['ts']}] {e['envelope']:8s} | {e['prompt_preview'][:80]}")


# Instantiate router in auto mode for the research pipeline (set to 'prompt' for interactive use)
router = CoCoSubagentRouter(session, MODEL, approval_mode='auto')
print('CoCoSubagentRouter ready. Approval mode:', router.approval_mode)

In [ ]:
class BiomedicalSubagent:
    """Base class for all biomedical research subagents."""

    system_prompt: str = ''
    envelope: str = 'RESEARCH'

    def __init__(self, session, model: str):
        self.session = session
        self.model = model

    def _complete(self, user_prompt: str, extra_system: str = '') -> str:
        system = self.system_prompt
        if extra_system:
            system = f'{system}\n\n{extra_system}'
        full = f'{system}\n\n{user_prompt}' if system else user_prompt
        escaped = full.replace("'", "''")
        row = self.session.sql(
            f"SELECT SNOWFLAKE.CORTEX.COMPLETE('{self.model}', '{escaped}')"
        ).collect()
        return row[0][0].strip() if row else ''

    def run(self, prompt: str, context: str = '') -> str:
        raise NotImplementedError


class PlannerAgent(BiomedicalSubagent):
    """Creates a structured research plan from a biomedical topic."""

    envelope = 'RESEARCH'
    system_prompt = textwrap.dedent("""
        You are a senior clinical research analyst.
        Given a biomedical research topic, produce a structured report plan.

        Output ONLY a JSON object with:
          - "title": the report title
          - "sections": list of objects, each with:
              - "name": section name
              - "queries": list of 2-3 specific search queries for this section

        Sections should include: Background, Clinical Presentation,
        Diagnostic Approach, Treatment Strategies, Outcomes & Prognosis.
        Output ONLY JSON, no markdown fences.
    """)

    def run(self, topic: str, context: str = '') -> dict:
        raw = self._complete(f'Research topic: {topic}')
        try:
            clean = raw.strip().lstrip('```json').lstrip('```').rstrip('```')
            return json.loads(clean)
        except json.JSONDecodeError:
            # Fallback plan
            return {
                'title': topic,
                'sections': [
                    {'name': 'Overview',    'queries': [topic]},
                    {'name': 'Treatment',   'queries': [f'{topic} treatment']},
                    {'name': 'Outcomes',    'queries': [f'{topic} outcomes']},
                ]
            }


class SearchAgent(BiomedicalSubagent):
    """Synthesises evidence from retrieved KB documents."""

    envelope = 'RO'
    system_prompt = textwrap.dedent("""
        You are a clinical evidence synthesiser.
        Given a search query and retrieved clinical records, produce a concise
        evidence summary (3-5 sentences) with specific findings.
        Cite the source type (e.g. Progress Note, Imaging Report, Lab Result).
        If no evidence is found, say 'No relevant data retrieved.'
    """)

    def run(self, query: str, context: str = '') -> str:
        user = f'Query: {query}\n\nRetrieved records:\n{context}' if context else query
        return self._complete(user)


class WriterAgent(BiomedicalSubagent):
    """Writes a report section from evidence summaries."""

    envelope = 'RW'
    system_prompt = textwrap.dedent("""
        You are a clinical report writer.
        Given a section name and evidence summaries, write a polished report
        section (3-5 paragraphs) suitable for a clinical research report.
        Use precise medical terminology. Reference source types in parentheses.
        Do NOT fabricate data not present in the evidence.
    """)

    def run(self, section_name: str, context: str = '') -> str:
        user = f'Section: {section_name}\n\nEvidence:\n{context}'
        return self._complete(user)


class ReflectorAgent(BiomedicalSubagent):
    """Identifies gaps in a draft report and generates follow-up queries."""

    envelope = 'RESEARCH'
    system_prompt = textwrap.dedent("""
        You are a critical research reviewer.
        Given a draft research report, identify:
        1. Missing information or evidence gaps
        2. Unsupported claims
        3. Sections needing more depth

        Output ONLY a JSON object with:
          - "gaps": list of gap descriptions
          - "follow_up_queries": list of specific queries to fill those gaps
        Output ONLY JSON, no markdown fences.
    """)

    def run(self, draft: str, context: str = '') -> dict:
        raw = self._complete(f'Draft report:\n{draft}')
        try:
            clean = raw.strip().lstrip('```json').lstrip('```').rstrip('```')
            return json.loads(clean)
        except json.JSONDecodeError:
            return {'gaps': ['Unable to parse reflection'], 'follow_up_queries': []}


class QAAgent(BiomedicalSubagent):
    """Answers freeform questions about a finished report."""

    envelope = 'RO'
    system_prompt = textwrap.dedent("""
        You are a clinical Q&A assistant.
        Answer questions about a research report accurately and concisely.
        Base your answer ONLY on the report provided.
        If the answer is not in the report, say so clearly.
    """)

    def run(self, question: str, context: str = '') -> str:
        user = f'Report:\n{context}\n\nQuestion: {question}'
        return self._complete(user)


# Instantiate all agents
planner   = PlannerAgent(session, MODEL)
searcher  = SearchAgent(session, MODEL)
writer    = WriterAgent(session, MODEL)
reflector = ReflectorAgent(session, MODEL)
qa_agent  = QAAgent(session, MODEL)

print('All biomedical subagents initialised.')

In [ ]:
def _fmt_rows(rows, cols) -> str:
    """Format Snowpark Row list into a readable text block."""
    if not rows:
        return '(no records found)'
    parts = []
    for r in rows:
        parts.append(' | '.join(f'{c}: {r[c]}' for c in cols if r[c]))
    return '\n'.join(parts)


def search_notes(query: str, limit: int = 4) -> str:
    """Full-text search over clinical progress notes."""
    q = query.replace("'", "''")
    rows = session.sql(f"""
        SELECT NOTE_TYPE, NOTE_TITLE, NOTE_TEXT, AUTHOR_NAME, SERVICE
        FROM {KB_DB}.{KB_SCHEMA}.NOTES
        WHERE NOTE_TEXT ILIKE '%{q}%'
           OR NOTE_TITLE ILIKE '%{q}%'
        LIMIT {limit}
    """).collect()
    return _fmt_rows(rows, ['NOTE_TYPE', 'NOTE_TITLE', 'NOTE_TEXT', 'SERVICE'])


def search_imaging(query: str, limit: int = 3) -> str:
    """Full-text search over imaging / radiology reports."""
    q = query.replace("'", "''")
    rows = session.sql(f"""
        SELECT REPORT_TEXT, IMPRESSION, FINDINGS, CLINICAL_INDICATION, RADIOLOGIST_NAME
        FROM {KB_DB}.{KB_SCHEMA}.IMAGING_REPORTS
        WHERE REPORT_TEXT ILIKE '%{q}%'
           OR IMPRESSION   ILIKE '%{q}%'
           OR FINDINGS     ILIKE '%{q}%'
        LIMIT {limit}
    """).collect()
    return _fmt_rows(rows, ['CLINICAL_INDICATION', 'IMPRESSION', 'FINDINGS'])


def search_problems(query: str, limit: int = 8) -> str:
    """Search the patient problem/diagnosis list."""
    q = query.replace("'", "''")
    rows = session.sql(f"""
        SELECT PROBLEM_NAME, STATUS, ONSET_DATE, PATIENT_ID
        FROM {KB_DB}.{KB_SCHEMA}.PROBLEMS
        WHERE PROBLEM_NAME ILIKE '%{q}%'
           OR PROBLEM_NAME ILIKE '%{q.split()[0]}%'
        LIMIT {limit}
    """).collect()
    return _fmt_rows(rows, ['PROBLEM_NAME', 'STATUS', 'ONSET_DATE'])


def search_labs(query: str, limit: int = 6) -> str:
    """Search lab results by test name."""
    q = query.replace("'", "''")
    rows = session.sql(f"""
        SELECT TEST_NAME, RESULT_VALUE, RESULT_UNIT, REFERENCE_RANGE, ABNORMAL_FLAG, COLLECTION_TIME
        FROM {KB_DB}.{KB_SCHEMA}.LAB_RESULTS
        WHERE TEST_NAME ILIKE '%{q}%'
           OR RESULT_VALUE::STRING ILIKE '%{q}%'
        LIMIT {limit}
    """).collect()
    return _fmt_rows(rows, ['TEST_NAME', 'RESULT_VALUE', 'RESULT_UNIT', 'ABNORMAL_FLAG'])


def multi_search(queries: list[str]) -> dict[str, str]:
    """Run searches across all KB tables in parallel — NVIDIA Parallel Search pattern.

    For each query, concurrently searches notes, imaging, problems, and labs,
    then synthesises each batch via SearchAgent.
    Returns dict: query → evidence_summary.
    """
    def _search_one(query: str) -> tuple[str, str]:
        # Gather raw hits from all four tables
        hits = {
            'Notes':   search_notes(query),
            'Imaging': search_imaging(query),
            'Problems': search_problems(query),
            'Labs':    search_labs(query),
        }
        combined = '\n\n'.join(f'[{src}]\n{txt}' for src, txt in hits.items()
                               if '(no records found)' not in txt)
        if not combined:
            combined = '(no records found across all tables)'
        # LLM-as-judge: synthesise into an evidence summary
        summary = searcher.run(query, context=combined)
        return query, summary

    results = {}
    with ThreadPoolExecutor(max_workers=min(len(queries), 4)) as pool:
        futures = {pool.submit(_search_one, q): q for q in queries}
        for future in as_completed(futures):
            q, summary = future.result()
            results[q] = summary
            print(f'  ✓ Searched: {q[:70]}')
    return results


# Quick smoke-test
print('Smoke-test — searching notes for "pneumonia":')
sample = search_notes('pneumonia', limit=1)
print(sample[:300])

In [ ]:
class BiomedicalResearchAgent:
    """Orchestrates the NVIDIA-blueprint deep research loop on Snowflake Cortex.

    Steps:
      1. Plan  — PlannerAgent (RESEARCH envelope)
      2. Search — parallel multi_search per section (RO envelope)
      3. Write  — WriterAgent per section (RW envelope)
      4. Reflect — ReflectorAgent on draft (RESEARCH envelope)
      5. Finalize — WriterAgent assembles final report (DEPLOY envelope)
    """

    def __init__(self, session, router: CoCoSubagentRouter, model: str):
        self.session  = session
        self.router   = router
        self.model    = model
        self._plan    = {}
        self._evidence: dict[str, dict] = {}
        self._sections: dict[str, str] = {}
        self._reflection = {}
        self.report   = ''

    # ── Step 1: Plan ─────────────────────────────────────────────────────────
    def step_plan(self, topic: str) -> dict:
        print('\n📝 Step 1 — Planning research structure...')
        self._plan = self.router.execute_with_envelope(
            lambda p: planner.run(p),
            topic,
            envelope='RESEARCH'
        )
        print(f"  Title: {self._plan.get('title', topic)}")
        for s in self._plan.get('sections', []):
            print(f"  Section: {s['name']}")
        return self._plan

    # ── Step 2: Parallel Search ───────────────────────────────────────────────
    def step_search(self) -> dict:
        print('\n🔍 Step 2 — Parallel search across KB tables...')
        sections = self._plan.get('sections', [])
        # Flatten all queries across sections
        all_queries = []
        for sec in sections:
            all_queries.extend(sec.get('queries', [sec['name']]))

        def _do_search(queries):
            return multi_search(queries)

        evidence_flat = self.router.execute_with_envelope(
            _do_search, all_queries, envelope='RO'
        )
        # Group evidence back by section
        for sec in sections:
            sec_queries = sec.get('queries', [sec['name']])
            self._evidence[sec['name']] = {
                q: evidence_flat.get(q, 'No evidence found.')
                for q in sec_queries
            }
        return self._evidence

    # ── Step 3: Write ─────────────────────────────────────────────────────────
    def step_write(self) -> dict:
        print('\n✍️  Step 3 — Writing report sections...')
        for sec_name, ev_dict in self._evidence.items():
            combined_evidence = '\n\n'.join(
                f'Query: {q}\nEvidence: {ev}'
                for q, ev in ev_dict.items()
            )
            section_text = self.router.execute_with_envelope(
                lambda context: writer.run(sec_name, context),
                combined_evidence,
                envelope='RW'
            )
            self._sections[sec_name] = section_text
            print(f'  ✓ Written: {sec_name}')
        return self._sections

    # ── Step 4: Reflect ───────────────────────────────────────────────────────
    def step_reflect(self) -> dict:
        print('\n🔄 Step 4 — Reflecting on gaps...')
        draft = '\n\n'.join(
            f'## {name}\n{text}' for name, text in self._sections.items()
        )
        self._reflection = self.router.execute_with_envelope(
            lambda p: reflector.run(p),
            draft,
            envelope='RESEARCH'
        )
        gaps = self._reflection.get('gaps', [])
        follow_ups = self._reflection.get('follow_up_queries', [])
        print(f'  Gaps identified: {len(gaps)}')
        for g in gaps:
            print(f'    • {g}')

        # Optionally run one more search pass on follow-up queries
        if follow_ups:
            print(f'  Running {len(follow_ups)} follow-up search(es)...')
            extra = multi_search(follow_ups[:3])  # cap at 3
            # Append follow-up evidence to 'Outcomes & Prognosis' or last section
            last_sec = list(self._sections.keys())[-1]
            extra_text = '\n\n'.join(f'{q}:\n{e}' for q, e in extra.items())
            self._sections[last_sec] += f'\n\n**Additional evidence (follow-up):**\n{extra_text}'

        return self._reflection

    # ── Step 5: Finalize ──────────────────────────────────────────────────────
    def step_finalize(self) -> str:
        print('\n📄 Step 5 — Finalising report...')
        title = self._plan.get('title', 'Biomedical Research Report')
        header = (
            f'# {title}\n'
            f'_Generated by BiomedicalResearchAgent on Snowflake Cortex_  \n'
            f'_Model: {self.model}_  \n'
            f'_Knowledge base: {KB_DB}.{KB_SCHEMA}_\n\n'
            '---\n'
        )
        body = '\n\n'.join(
            f'## {name}\n{text}' for name, text in self._sections.items()
        )
        footer = (
            '\n\n---\n'
            '### Sources\n'
            f'- `{KB_DB}.{KB_SCHEMA}.NOTES`\n'
            f'- `{KB_DB}.{KB_SCHEMA}.IMAGING_REPORTS`\n'
            f'- `{KB_DB}.{KB_SCHEMA}.PROBLEMS`\n'
            f'- `{KB_DB}.{KB_SCHEMA}.LAB_RESULTS`\n'
        )
        self.report = header + body + footer
        return self.report

    # ── Full pipeline ─────────────────────────────────────────────────────────
    def run(self, topic: str) -> str:
        t0 = time.time()
        self.step_plan(topic)
        self.step_search()
        self.step_write()
        self.step_reflect()
        report = self.step_finalize()
        elapsed = time.time() - t0
        print(f'\n✅ Research complete in {elapsed:.1f}s')
        self.router.audit_summary()
        return report


agent = BiomedicalResearchAgent(session, router, MODEL)
print('BiomedicalResearchAgent ready.')

In [ ]:
RESEARCH_TOPIC = 'MRSA pneumonia management and mechanical ventilation weaning in ICU patients'

# Check routing decision first (illustrates subagent-cortex-code routing)
routing = router.route(RESEARCH_TOPIC)
print('Routing decision:', json.dumps(routing, indent=2))

In [ ]:
# Execute the full deep research pipeline
report = agent.run(RESEARCH_TOPIC)

In [ ]:
import re

def _md_to_html(md: str) -> str:
    """Minimal markdown-to-HTML (headings, bold, italic, paragraphs)."""
    lines = md.split('\n')
    html_lines = []
    for line in lines:
        # Headings
        m = re.match(r'^(#{1,6})\s+(.*)', line)
        if m:
            level = len(m.group(1))
            html_lines.append(f'<h{level}>{m.group(2)}</h{level}>')
            continue
        # Bold / italic
        line = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', line)
        line = re.sub(r'\*(.+?)\*', r'<em>\1</em>', line)
        # Bullet lists
        if re.match(r'^\s*[-*]\s+', line):
            line = '<li>' + re.sub(r'^\s*[-*]\s+', '', line) + '</li>'
        # Blank line → paragraph break
        if not line.strip():
            html_lines.append('<br>')
        else:
            html_lines.append(line)
    return '\n'.join(html_lines)

html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Research Report</title>
<style>body{{font-family:sans-serif;max-width:800px;margin:40px auto;padding:0 20px;line-height:1.6}}</style>
</head><body>
{_md_to_html(report)}
</body></html>"""

with open('report.html', 'w') as f:
    f.write(html)
print(f'Wrote report.html ({len(html)} chars)')

In [ ]:
# Interactive Q&A — HITL pattern from NVIDIA blueprint
# Note: input() is not supported in Snowflake Notebooks, so we use a
# predefined question list instead of an interactive loop.

print('='*60)
print('Biomedical Research Q&A')
print('Envelope: RO | Model:', MODEL)
print('='*60)

# Replace or extend this list with your own questions
questions = [
    'What are the key molecular targets identified in the report?',
]

for question in questions:
    print(f'\nQ: {question}')
    answer = router.execute_with_envelope(
        lambda q: qa_agent.run(q, context=report),
        question,
        envelope='RO'
    )
    print(f'\nAnswer:\n{answer}')

In [ ]:
from IPython.display import display, HTML
display(HTML(html))